In [1]:
import pandas as pd

In [2]:
beindata = pd.read_csv(r"s:\16.10.25\26969870_BEINDATANEWRPT.CSV",dtype='str', on_bad_lines='skip')

ft = pd.read_csv(r"s:\16.10.25\26967761_NEWCNEFINTRANSRPT.CSV",dtype='str',on_bad_lines='skip')
swap_rep = pd.read_csv(r"s:\16.10.25\26969408_SWAPRPT.CSV",dtype='str', on_bad_lines='skip')

swap_boxes = pd.read_csv(r"S:\1000s project\1000s for swap.csv",dtype='str', on_bad_lines='skip')

swap_rep=swap_rep[['Old Serial Number','New Serial Number','Received Entity','Swap Date']]
swapped = swap_boxes.merge(right=swap_rep,left_on='Decoder', right_on="Old Serial Number")
swapped['Status'] = 'Swapped'

bein_active = beindata.loc[(beindata['Status']=='Active')]
bein_active= bein_active[['Customer Number','Customer Type','Smart Card','Decoder','Item Description STB']]
active = swap_boxes.loc[swap_boxes['Customer Number'].isin(bein_active['Customer Number'])]
renewed = active.loc[~active['Customer Number'].isin(swapped['Customer Number'])].copy()
renewed['Status'] = "Active to be swapped"


In [3]:

ft_renewed = ft.copy()
ft_renewed = ft_renewed.merge(beindata[['Customer Number','Smart Card','Decoder']],left_on=['Subscriber Nr', 'Smartcard'], right_on=['Customer Number','Smart Card'])

ft_renewed['Amount'] = pd.to_numeric(ft_renewed['Amount'])
ft_renewed['Created Date'] = pd.to_datetime(ft_renewed['Created Date'],dayfirst=True)

ft_renewed = ft_renewed.loc[(ft_renewed["Doc Type"].isin(["JV","Payment"])) & ft_renewed['Subscriber Nr'].isin(renewed['Customer Number']) & (ft_renewed['Amount']>200)]
ft_renewed = ft_renewed.sort_values(['Smartcard','Created Date'],ascending=[True,False])

renewed =  renewed.merge(right=ft_renewed[['Decoder','Created Date']],right_on='Decoder', left_on="Decoder",how='left')

renewed = renewed.rename({"Created Date":"Swap Date"},axis=1)
renewed = renewed.drop_duplicates(subset=["Customer Number"])



result = pd.concat([swapped,renewed])

result['Swap Date'] = pd.to_datetime(result['Swap Date'],dayfirst=True, errors="coerce")
start_date = pd.to_datetime("20-09-2025", dayfirst=True)
result["week"] = "w" + (((result["Swap Date"] - start_date).dt.days // 7) + 1).astype("Int64").astype(str)
result = result.drop_duplicates(subset=['Customer Number'])
result.loc[result['Received Entity']=='Force Swap_Maadi','Received Entity'] = 'Maadi showroom'
result.loc[result['Received Entity']=='Force Swap_Mohandseen','Received Entity'] = 'Mohandeseen Showroom'

result = result.merge(right=beindata[['Decoder','Item Description STB']], left_on='New Serial Number' , right_on="Decoder", how='left')
result = result.loc[~result['Item Description STB'].str.contains('1000',na=False)]

#result = result.loc[~result['Customer Number'].isin(exclude['Customer Number'])]



C:\Users\mturky\AppData\Local\Temp\1\ipykernel_3944\3727418878.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ft_renewed['Created Date'] = pd.to_datetime(ft_renewed['Created Date'],dayfirst=True)
C:\Users\mturky\AppData\Local\Temp\1\ipykernel_3944\3727418878.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  result['Swap Date'] = pd.to_datetime(result['Swap Date'],dayfirst=True, errors="coerce")


In [4]:
result = result.drop_duplicates()

In [5]:
result.to_csv(f'1000s_swap.csv',index=False)

In [6]:
bein_active.loc[bein_active['Decoder']=='0372296751']

,Customer Number,Customer Type,Smart Card,Decoder,Item Description STB
11803,3198676,CNE Subscriber,10732857783,0372296751,beIN 4k


In [7]:
beindata.loc[beindata['Customer Number']=='15250851']

,Customer Number,Customer Type,Entity,Contract Number,Start Date,End Date,Plan,Status,Decoder,Item Description STB,Smart Card,Item Description SC,Next Billing Date,Billing Cycle,PPV Balance,Customer Balance,Outstanding Balance
2123979,15250851,beIN Quartar Installment,CNE Head office,239061,08-05-2019,07-05-2020,beIN Premium,DIS,0372307131,beIN 4k,10732853501,CNE V7 Card,08-05-2020,12M,0 Dr,0 Dr,0 DR
2123980,15250851,beIN Quartar Installment,CNE Head office,1334738,22-11-2020,21-11-2021,beIN Premium,DIS,0372307131,beIN 4k,10732853501,CNE V7 Card,22-11-2021,3M,0 Dr,0 Dr,0 DR
2123981,15250851,beIN Quartar Installment,CNE Head office,3819066,02-10-2025,01-10-2026,PREMIUM 09.24,Active,0372307131,beIN 4k,10732853501,CNE V7 Card,02-01-2026,3M,0 Dr,0 Dr,0 DR


In [8]:
beindata.loc[beindata['Decoder']=='0314722965']

,Customer Number,Customer Type,Entity,Contract Number,Start Date,End Date,Plan,Status,Decoder,Item Description STB,Smart Card,Item Description SC,Next Billing Date,Billing Cycle,PPV Balance,Customer Balance,Outstanding Balance
2332479,14583006,beIN Quartar Installment,El Rehab,995268,15-11-2019,14-11-2020,beIN Sports,DIS,0314722965,beIN Decoder 1000s,10110677654,beIN Smartcard 1000s,15-11-2020,3M,0 Dr,.01 Cr,.01 CR
2332480,14583006,beIN Quartar Installment,El Rehab,172887,15-11-2018,14-11-2019,beIN Sports,DIS,0314722965,beIN Decoder 1000s,10110677654,beIN Smartcard 1000s,15-11-2019,3M,0 Dr,.01 Cr,.01 CR
2332481,14583006,beIN Quartar Installment,El Rehab,1329311,16-11-2020,15-11-2021,beIN Sports,DIS,0314722965,beIN Decoder 1000s,10110677654,beIN Smartcard 1000s,16-11-2021,3M,0 Dr,.01 Cr,.01 CR
2332482,14583006,beIN Quartar Installment,El Rehab,3827761,09-10-2025,08-10-2026,TOGETHER 09.24,Active,0314722965,beIN Decoder 1000s,10110677654,beIN Smartcard 1000s,09-01-2026,3M,0 Dr,.01 Cr,.01 CR


In [9]:
display(swap_boxes.loc[swap_boxes['Decoder']=='0302755221'])
display(swap_boxes.loc[swap_boxes['Smart Card']=='42916839501'])

,Customer Number,Customer Type,End Date,Decoder,Smart Card
7658,12954936,CNE Subscriber,11/6/2014,0302755221,42916839501


,Customer Number,Customer Type,End Date,Decoder,Smart Card
7658,12954936,CNE Subscriber,11/6/2014,0302755221,42916839501


In [10]:
ft1.loc[ft1['Subscriber Nr'] == '14583006']

NameError: name 'ft1' is not defined

In [ ]:
ft.loc[ft['Subscriber Nr'] == '15250851']

,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,...,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,Payment Flag
276778,15250851,Debit Note,DR_724608,02/10/2025 12:00:00 AM,04:01:23 PM,Posted,NaN,NaN,NaN,0.01,...,NaN,NaN,NaN,NaN,NaN,beIN,beIN Quartar Installment,CNE Head office,NaN,NaN
276895,15250851,Invoice,INV_8332445,02/10/2025 12:00:00 AM,03:58:55 PM,Posted,22/08/2021 12:00:00 AM,21/11/2021 12:00:00 AM,NaN,684,...,22/08/2021 - 21/11/2021,3M,Subscription Invoice,beIN Premium,1334738,beIN,beIN Quartar Installment,CNE Head office,NaN,NaN
418015,15250851,Payment,CNEPMT_2118663,02/10/2025 12:00:00 AM,03:58:54 PM,Posted,NaN,NaN,NaN,2206,...,NaN,NaN,NaN,NaN,NaN,beIN,beIN Quartar Installment,CNE Head office,2083,Normal payment
699878,15250851,Debit Note,DR_724611,02/10/2025 12:00:00 AM,04:16:11 PM,Posted,NaN,NaN,NaN,0.01,...,NaN,NaN,NaN,NaN,NaN,beIN,beIN Quartar Installment,CNE Head office,NaN,NaN
841116,15250851,Credit Note,CR_724610,02/10/2025 12:00:00 AM,04:15:39 PM,Posted,NaN,NaN,NaN,0.01,...,NaN,NaN,NaN,NaN,NaN,beIN,beIN Quartar Installment,CNE Head office,NaN,NaN
951648,15250851,Credit Note,CR_510160,22/09/2025 12:00:00 AM,08:59:44 AM,Posted,NaN,NaN,NaN,0.01,...,NaN,NaN,NaN,NaN,NaN,beIN,beIN Quartar Installment,CNE Head office,NaN,NaN
1123481,15250851,Invoice,INV_8332449,02/10/2025 12:00:00 AM,03:59:25 PM,Posted,02/10/2025 12:00:00 AM,01/01/2026 12:00:00 AM,NaN,1522,...,02/10/2025 - 01/01/2026,3M,Subscription Invoice,PREMIUM 09.24,3819066,beIN,beIN Quartar Installment,CNE Head office,NaN,NaN


In [12]:
result.loc[result['Customer Number']=='16587332']

,Customer Number,Customer Type,End Date,Decoder_x,Smart Card,Old Serial Number,New Serial Number,Received Entity,Swap Date,Status,week,Decoder_y,Item Description STB
136,16587332,CNE Subscriber,17/9/2017,0319670279,10388233214,0319670279,0373041239,Maadi showroom,2025-10-13,Swapped,w4,0373041239,beIN 4k
